# Chapter 9 Companion Notebook: Credit Risk Modeling

This notebook reproduces every worked numerical example from Chapter 9 of *AI in Finance* (`chapter9.tex`): expected loss, collateral-coverage LGD regression, credit stress testing, credit scoring via logistic regression, discriminatory-power validation (AUC/Gini), the Merton structural credit model, CDS-implied default probabilities, rating transition matrices, portfolio credit risk / economic capital, the single-factor (Vasicek) correlated-default model, and operational risk capital.

## 1. Credit risk: expected loss (Section 9.2)

In [1]:
import numpy as np
from scipy.stats import norm

PD, LGD, EAD = 0.03, 0.60, 1000.0
expected_loss = PD * LGD * EAD
print(f"Expected loss per bond: ${expected_loss:.2f}")
print(f"Check against Chapter 5: promised $1,000 - expected payoff $982 = ${1000-982}")

Expected loss per bond: $18.00
Check against Chapter 5: promised $1,000 - expected payoff $982 = $18


## 1b. What drives LGD: a collateral-coverage regression (Section 9.2.1)

In [2]:
coverage = np.array([0.40, 0.60, 0.80, 1.20])
recovery = np.array([0.25, 0.45, 0.55, 0.75])

xbar, ybar = coverage.mean(), recovery.mean()
b_hat = np.sum((coverage-xbar)*(recovery-ybar)) / np.sum((coverage-xbar)**2)
a_hat = ybar - b_hat*xbar
print(f"a={a_hat:.3f}, b={b_hat:.3f}")

pred = a_hat + b_hat*coverage
resid = recovery - pred
r2 = 1 - np.sum(resid**2)/np.sum((recovery-ybar)**2)
print(f"predicted: {pred}, R^2={r2:.4f}")

pred_at_1 = a_hat + b_hat*1.0
print(f"Predicted recovery at coverage=1.0: {pred_at_1:.2f} (implied LGD={1-pred_at_1:.2f})")

a=0.050, b=0.600
predicted: [0.29 0.41 0.53 0.77], R^2=0.9692
Predicted recovery at coverage=1.0: 0.65 (implied LGD=0.35)


## 1c. Credit stress testing: stressed PD and LGD (Section 9.2.2)

In [3]:
PD_stress, LGD_stress = 0.08, 0.70
EL_base = PD*LGD*EAD
EL_stress = PD_stress*LGD_stress*EAD
print(f"EL base: ${EL_base:.2f}")
print(f"EL stress: ${EL_stress:.2f}")
print(f"Multiple: {EL_stress/EL_base:.3f}x")

EL base: $18.00
EL stress: $56.00
Multiple: 3.111x


## 2. Credit scoring with logistic regression (Section 9.3)

In [4]:
from sklearn.linear_model import LogisticRegression

X_loans = np.array([
    [35, 2], [20, 8], [40, 1], [15, 10], [30, 3], [18, 7],
], dtype=float)
y_loans = np.array([1, 0, 1, 0, 1, 0])  # 1 = default
names = ['A','B','C','D','E','F']

model = LogisticRegression()
model.fit(X_loans, y_loans)
print(f"Coefficients: {model.coef_}, Intercept: {model.intercept_}")

new_applicant = np.array([[28, 4]])
prob_default = model.predict_proba(new_applicant)[0, 1]
print(f"P(default) for new applicant: {prob_default:.4f}")

score = 850 - 550 * prob_default
print(f"Credit score: {score:.0f}")

Coefficients: [[ 0.52992476 -0.23135204]], Intercept: [-12.15176948]
P(default) for new applicant: 0.8533
Credit score: 381


## 2b. Validating a credit scoring model: AUC and Gini (Section 9.3.1)

In [5]:
probs = model.predict_proba(X_loans)[:, 1]
for nm, p, y in zip(names, probs, y_loans):
    print(f"{nm}: p={p:.5f} y={y}")

defaults = probs[y_loans == 1]
nondefaults = probs[y_loans == 0]
concordant = sum(1.0 if pd_ > pn else (0.5 if pd_ == pn else 0.0) for pd_ in defaults for pn in nondefaults)
total = len(defaults) * len(nondefaults)
auc = concordant / total
gini = 2*auc - 1
print(f"\nAUC={auc:.4f} Gini={gini:.4f} (concordant {concordant}/{total})")

A: p=0.99736 y=1
B: p=0.03217 y=0
C: p=0.99985 y=1
D: p=0.00148 y=0
E: p=0.95487 y=1
F: p=0.01431 y=0

AUC=1.0000 Gini=1.0000 (concordant 9.0/9)


## 3. Structural credit models: the Merton model (Section 9.4)

In [6]:
V0, D, r, sigma_V, T = 150.0, 100.0, 0.04, 0.25, 1.0

d1 = (np.log(V0 / D) + (r + sigma_V**2 / 2) * T) / (sigma_V * np.sqrt(T))
d2 = d1 - sigma_V * np.sqrt(T)
E0 = V0 * norm.cdf(d1) - D * np.exp(-r * T) * norm.cdf(d2)
PD_merton = norm.cdf(-d2)

print(f"d1 = {d1:.4f}, d2 = {d2:.4f}")
print(f"Implied equity value: ${E0:.2f} million")
print(f"Merton one-year default probability: {PD_merton:.4%}")

d1 = 1.9069, d2 = 1.6569
Implied equity value: $54.37 million
Merton one-year default probability: 4.8774%


## 3b. Market-implied default probabilities: CDS spreads (Section 9.4.1)

In [7]:
s_cds, R_cds = 0.02, 0.40
h = s_cds / (1 - R_cds)
pd_1yr = 1 - np.exp(-h)
print(f"Hazard rate: {h:.4%}")
print(f"1-year default probability: {pd_1yr:.4%}")

Hazard rate: 3.3333%
1-year default probability: 3.2784%


## 4. Rating models and transition matrices (Section 9.5)

In [8]:
P = np.array([
    [0.90, 0.08, 0.02],
    [0.10, 0.80, 0.10],
    [0.00, 0.00, 1.00],
])

P2 = P @ P
P3 = P2 @ P
print("One-year PD -- High: {:.0%}, Medium: {:.0%}".format(P[0, 2], P[1, 2]))
print(f"Two-year PD -- High: {P2[0, 2]:.1%}, Medium: {P2[1, 2]:.1%}")
print(f"Three-year PD -- High: {P3[0, 2]:.3%}")

One-year PD -- High: 2%, Medium: 10%
Two-year PD -- High: 4.6%, Medium: 18.2%
Three-year PD -- High: 7.596%


## 5. Portfolio credit risk and economic capital (Section 9.6)

In [9]:
n_loans = 100
EL_portfolio = n_loans * expected_loss
print(f"Portfolio expected loss: ${EL_portfolio:,.2f}")

std_defaults = np.sqrt(n_loans * PD * (1 - PD))
std_loss = std_defaults * LGD * EAD
print(f"Std of dollar loss: ${std_loss:,.2f}")

z99 = norm.ppf(0.99)
VaR99_losses = EL_portfolio + z99 * std_loss
economic_capital = z99 * std_loss
print(f"99% VaR of portfolio losses: ${VaR99_losses:,.2f}")
print(f"Economic capital: ${economic_capital:,.2f}")
print(f"Ratio of economic capital to expected loss: {economic_capital/EL_portfolio:.2f}x")

Portfolio expected loss: $1,800.00
Std of dollar loss: $1,023.52
99% VaR of portfolio losses: $4,181.07
Economic capital: $2,381.07
Ratio of economic capital to expected loss: 1.32x


## 5b. Correlated defaults: the single-factor (Vasicek) model (Section 9.6.1)

In [10]:
def Ninv(p):
    return norm.ppf(p)

rho, alpha = 0.20, 0.99

def WCDR(PD_, rho_=rho, alpha_=alpha):
    return norm.cdf((Ninv(PD_) + np.sqrt(rho_)*Ninv(alpha_)) / np.sqrt(1-rho_))

wcdr_base = WCDR(PD)
loss_corr = n_loans * wcdr_base * LGD * EAD
ec_corr = loss_corr - EL_portfolio
print(f"WCDR(PD={PD}): {wcdr_base:.4%}")
print(f"Correlated worst-case loss: ${loss_corr:,.2f}, economic capital: ${ec_corr:,.2f}")
print(f"Ratio to independent EC ({economic_capital:.2f}): {ec_corr/economic_capital:.2f}x")

# Combined stress: PD=8%, rho=0.20
PD_s = 0.08
EL_s = n_loans*PD_s*LGD*EAD
wcdr_s = WCDR(PD_s)
loss_corr_s = n_loans*wcdr_s*LGD*EAD
ec_corr_s = loss_corr_s - EL_s
print(f"\nStressed PD={PD_s}: WCDR={wcdr_s:.4%}, EL={EL_s:.2f}, loss={loss_corr_s:.2f}, EC={ec_corr_s:.2f}")
print(f"Ratio of stressed-correlated EC to base independent EC: {ec_corr_s/economic_capital:.2f}x")

WCDR(PD=0.03): 17.3707%
Correlated worst-case loss: $10,422.42, economic capital: $8,622.42
Ratio to independent EC (2381.07): 3.62x

Stressed PD=0.08: WCDR=34.1731%, EL=4800.00, loss=20503.85, EC=15703.85
Ratio of stressed-correlated EC to base independent EC: 6.60x


## 6. Operational risk capital: the Basic Indicator Approach (Section 9.6.3)

In [11]:
avg_gross_income = 200_000_000
alpha_op = 0.15
op_risk_capital = alpha_op * avg_gross_income
print(f"Operational risk capital: ${op_risk_capital:,.0f}")

Operational risk capital: $30,000,000


## Exercises (match Chapter 9, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [12]:
# Exercise 1: expected loss for a $500,000 loan, PD=5%, LGD=50%
EL_ex1 = 0.05 * 0.50 * 500_000
print(f"Exercise 1 -- Expected loss: ${EL_ex1:,.0f}")

# Exercise 2: LGD regression prediction at coverage=1.50
pred_ex2 = a_hat + b_hat*1.50
print(f"Exercise 2 -- Predicted recovery at coverage=1.50: {pred_ex2:.2f} (implied LGD={1-pred_ex2:.2f})")

# Exercise 3: credit score for debt/income=22%, credit history=6 years
new_applicant_ex3 = np.array([[22, 6]])
prob_ex3 = model.predict_proba(new_applicant_ex3)[0, 1]
score_ex3 = 850 - 550 * prob_ex3
print(f"Exercise 3 -- P(default)={prob_ex3:.4f}, Score={score_ex3:.0f}")

# Exercise 4: milder stress PD=6%, LGD=65%
EL_ex4 = 0.06*0.65*EAD
print(f"Exercise 4 -- EL stress (milder): ${EL_ex4:.2f}, multiple={EL_ex4/EL_base:.3f}x")

# Exercise 6: four-year cumulative PD for Medium-rated borrower
P4 = P3 @ P
print(f"Exercise 6 -- Four-year PD (Medium): {P4[1, 2]:.3%}")

# Exercise 7: Merton model for a different firm
V0_ex, D_ex, r_ex, sigma_V_ex, T_ex = 200.0, 120.0, 0.05, 0.30, 1.0
d1_ex = (np.log(V0_ex / D_ex) + (r_ex + sigma_V_ex**2 / 2)*T_ex) / (sigma_V_ex*np.sqrt(T_ex))
d2_ex = d1_ex - sigma_V_ex*np.sqrt(T_ex)
PD_ex = norm.cdf(-d2_ex)
print(f"Exercise 7 -- Merton PD: {PD_ex:.4%}")

# Exercise 8: 200-loan portfolio, EAD=2000, PD=4%, LGD=45%
n_ex8, EAD_ex8, PD_ex8, LGD_ex8 = 200, 2000.0, 0.04, 0.45
EL_ex8 = n_ex8*PD_ex8*LGD_ex8*EAD_ex8
std_ex8 = np.sqrt(n_ex8*PD_ex8*(1-PD_ex8))*LGD_ex8*EAD_ex8
ec_ex8 = z99*std_ex8
print(f"Exercise 8 -- EL=${EL_ex8:,.2f}, std=${std_ex8:,.2f}, EC=${ec_ex8:,.2f}")

# Exercise 9: operational risk capital, gross income=$350M
op_ex9 = 0.15 * 350_000_000
print(f"Exercise 9 -- Operational risk capital: ${op_ex9:,.0f}")

# Exercise 15: Vasicek WCDR with rho=0.12
wcdr_ex15 = WCDR(PD, rho_=0.12)
print(f"Exercise 15 -- WCDR(rho=0.12): {wcdr_ex15:.4%} (vs rho=0.20: {wcdr_base:.4%})")

# Exercise 16: CDS spread=350bp, recovery=35%
h_ex16 = 0.035/(1-0.35)
pd_ex16 = 1-np.exp(-h_ex16)
print(f"Exercise 16 -- hazard rate={h_ex16:.4%}, PD 1yr={pd_ex16:.4%}")

# Exercise 17: combined PD=6%, rho=0.15
wcdr_ex17 = WCDR(0.06, rho_=0.15)
EL_ex17 = n_loans*0.06*LGD*EAD
loss_ex17 = n_loans*wcdr_ex17*LGD*EAD
ec_ex17 = loss_ex17 - EL_ex17
print(f"Exercise 17 -- WCDR={wcdr_ex17:.4%}, EL=${EL_ex17:,.2f}, EC=${ec_ex17:,.2f}")

Exercise 1 -- Expected loss: $12,500
Exercise 2 -- Predicted recovery at coverage=1.50: 0.95 (implied LGD=0.05)
Exercise 3 -- P(default)=0.1322, Score=777
Exercise 4 -- EL stress (milder): $39.00, multiple=2.167x
Exercise 6 -- Four-year PD (Medium): 30.776%
Exercise 7 -- Merton PD: 4.2769%
Exercise 8 -- EL=$7,200.00, std=$2,494.15, EC=$5,802.27
Exercise 9 -- Operational risk capital: $52,500,000
Exercise 15 -- WCDR(rho=0.12): 12.5924% (vs rho=0.20: 17.3707%)
Exercise 16 -- hazard rate=5.3846%, PD 1yr=5.2422%
Exercise 17 -- WCDR=23.9123%, EL=$3,600.00, EC=$10,747.37
